In [3]:
from ipynb.fs.defs.agent import DQNAgent
from ipynb.fs.defs.trading_env import TradingEnv
from ipynb.fs.defs.data import reinforcement_data
from ipynb.fs.defs.features import build_features
from ipynb.fs.defs.data import reinforcement_data
   
from ipynb.fs.defs.save_load import save_model
from ipynb.fs.defs.save_load import load_model
import random

In [4]:
stocks = [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META"
]

feature_cache = {}

for ticker in stocks:

    prices = reinforcement_data(
        ticker,
        "2020-01-01",
        "2024-01-01"
    ).squeeze()

    feature_cache[ticker] = build_features(prices)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [5]:
agent = DQNAgent()
episodes = 500

for episode in range(episodes):

    ticker = random.choice(stocks)

    env = TradingEnv(feature_cache[ticker])

    state = env.reset()
    done = False

    episode_reward = 0

    while not done:

        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)

        agent.replay_buffer.push(
            (
                state,
                action,
                reward,
                next_state,
                done
            )
        )

        agent.train_step()

        state = next_state

        episode_reward += reward

    agent.epsilon = max(agent.epsilon_min,agent.epsilon* agent.epsilon_decay)

    agent.update_target_net()

    print(
        f"Episode {episode+1}/{episodes} "
        f"| Reward: {episode_reward:.4f} "
        f"| Epsilon: {agent.epsilon:.4f}"
    )

Episode 1/500 | Reward: 10175107.2972 | Epsilon: 0.9900
Episode 2/500 | Reward: 10569631.1883 | Epsilon: 0.9801
Episode 3/500 | Reward: 10532173.2351 | Epsilon: 0.9703
Episode 4/500 | Reward: 13953417.3532 | Epsilon: 0.9606
Episode 5/500 | Reward: 17310207.9631 | Epsilon: 0.9510
Episode 6/500 | Reward: 12193163.1268 | Epsilon: 0.9415
Episode 7/500 | Reward: 11732769.9929 | Epsilon: 0.9321
Episode 8/500 | Reward: 11733867.3079 | Epsilon: 0.9227
Episode 9/500 | Reward: 15633248.5705 | Epsilon: 0.9135
Episode 10/500 | Reward: 12609079.6533 | Epsilon: 0.9044
Episode 11/500 | Reward: 11789133.7141 | Epsilon: 0.8953
Episode 12/500 | Reward: 12837535.5968 | Epsilon: 0.8864
Episode 13/500 | Reward: 12363916.9300 | Epsilon: 0.8775
Episode 14/500 | Reward: 14267498.9873 | Epsilon: 0.8687
Episode 15/500 | Reward: 12001147.8788 | Epsilon: 0.8601
Episode 16/500 | Reward: 13574551.0460 | Epsilon: 0.8515
Episode 17/500 | Reward: 11607964.5576 | Epsilon: 0.8429
Episode 18/500 | Reward: 14839806.2364 |

KeyboardInterrupt: 

In [6]:
save_model(agent,filepath="models/dqn_trader.pt",episodes=episodes)

Model saved to models/dqn_trader.pt
